# Compare model providers and judge replies: practice

Generate a shared question → collect competitors and answers → assemble numbered replies → ask a judge → parse JSON → map one-based ranks to list indices.

Use this with [the walkthrough](walkthrough.md).

**Source:**

- [1_foundations/2_lab2.ipynb](../../../../../1_foundations/2_lab2.ipynb)

**Run one cell at a time:** predict → run with Shift+Enter → explain → change one thing. Select the root project .venv kernel.

**Offline adaptation:** code below uses fictional data or local libraries. It never executes the linked source, reads a credential file, sends messages, starts a UI/server, or changes the course databases. Simulations are labeled; they do not verify external integrations.

## 1. Configuration stays outside executable examples

Environment-variable names identify settings. Values belong in .environment/.env and are loaded deliberately. This exercise inspects only the path and simulated presence flags; it reads no private file or variable values.

**Predict:** Does finding a file prove that authentication works?

In [ ]:
from pathlib import Path
# Find the repository from a kernel started inside it.
root = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "1_foundations" / "3_lab3.ipynb").is_file()), None)
if root is None:
    raise FileNotFoundError("Start your kernel inside ai_agents")
configuration_path = root / ".environment" / ".env"
print("Configuration path:", configuration_path)
simulated_presence = {"OPENAI_API_KEY": True, "GOOGLE_API_KEY": False}
print(simulated_presence)  # Booleans only; these are fictional flags.
assert configuration_path.name == ".env"

**Check:** No. File existence, variable presence, and successful provider authentication are separate checks.

**Change one thing:** Explain the difference between a missing file, a missing variable, and a provider rejecting authentication. Do not print token prefixes or values.

## 2. Provider endpoint and model selection

A compatible client can target different services. The endpoint selects the provider and its credentials; the model identifier selects a model there. Compatibility does not guarantee every feature works.

**Predict:** Does changing only the variable name change the provider?

In [ ]:
# Non-secret connection descriptions only; no clients or keys are created.
providers = {
    "openai": {"key_variable": "OPENAI_API_KEY", "endpoint": "default"},
    "gemini": {"key_variable": "GOOGLE_API_KEY", "endpoint": "https://generativelanguage.googleapis.com/v1beta/openai/"},
    "ollama": {"key_variable": None, "endpoint": "http://localhost:11434/v1"},
}
for provider, settings in providers.items():
    print(provider, settings["endpoint"], settings["key_variable"])
assert providers["gemini"]["key_variable"] == "GOOGLE_API_KEY"

**Check:** No. The configured endpoint/client and model select the service. Local Ollama still needs its service and downloaded model.

**Change one thing:** Add a provider from the linked source. Copy its endpoint and variable name, never a credential value.

## 3. Messages: instructions, earlier turns, and the new question

The model request is a list of messages. A role says who is speaking; content holds the text. The system instruction and prior conversation serve different purposes.

**Predict:** How many messages are sent on the follow-up turn?

In [ ]:
instructions = "Explain Python with small examples."
history = [
    {"role": "user", "content": "What is a list?"},
    {"role": "assistant", "content": "An ordered collection of values."},
]
# List addition preserves the order: instruction, old turns, new question.
messages = [{"role": "system", "content": instructions}] + history + [
    {"role": "user", "content": "How is a dictionary different?"}
]
print([item["role"] for item in messages])
assert len(messages) == 4

**Check:** Four: system, user, assistant, user. With empty history, only two remain.

**Change one thing:** Set history to an empty list. Explain what is lost while the system instruction remains.

## 4. Compare alternatives and decode a ranking

Parallel candidates answer the same question. A separate selection step chooses among them. A sample JSON result lets you practice decoding without trusting a model judge.

**Predict:** Why subtract one from a competitor number?

In [ ]:
import json
competitors = ["writer_a", "writer_b", "writer_c"]
answers = ["Detailed answer", "Short answer", "Answer with an example"]
assert len(competitors) == len(answers)
for label, answer in zip(competitors, answers):
    print(label, "→", answer)
ranking = json.loads('{"results": ["3", "1", "2"]}')
ranks = [int(value) for value in ranking["results"]]
# Validate that every candidate appears exactly once.
assert sorted(ranks) == list(range(1, len(competitors) + 1))
ordered = [competitors[number - 1] for number in ranks]
print("Ranking:", ordered)
assert ordered[0] == "writer_c"

**Check:** The judge uses one-based labels; list indices start at zero. A ranking is a judgment, not an objective accuracy proof.

**Change one thing:** Reorder the sample ranking and update the expected winner. Try a duplicate rank and see the validation reject it.

## 5. Structured output validates shape

Pydantic models name the fields you expect and check constraints. This helps callers use results reliably, but a valid field can still contain a false claim.

**Predict:** Will confidence=2 pass the constraint?

In [ ]:
from pydantic import BaseModel, Field, ValidationError
class Report(BaseModel):
    summary: str
    confidence: float = Field(ge=0, le=1)
report = Report(summary="Fictional result", confidence=0.8)
print(report.model_dump())
try:
    Report(summary="Bad shape", confidence=2)
except ValidationError:
    print("Invalid confidence rejected")
else:
    raise AssertionError("Constraint was not enforced")

**Check:** No. The value must be between zero and one. Validation checks the constraint, not the truth of the summary.

**Change one thing:** Add a field named follow_up_questions with type list[str]. Construct a valid object using it.

## Rebuild from memory

1. Recreate the main data-flow step in a new scratch cell.
2. Add one case that should fail and identify its exception/stop condition.
3. Compare your version with the linked source cells or functions.
4. Explain where a real model or service would enter, and what it could change.

For deliberate original-source execution, see the shared [configuration guide](../../../configuration.md). All provider/transport integrations remain UNVERIFIED at runtime.